# Урок 22. Списки и их методы

10 класс · III четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/index.ipynb) · [← Урок 21](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-21.ipynb) · [Урок 23 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-23.ipynb)

---

Методы списка, срезы, копирование и подводные камни ссылок. Генераторы списков. Сортировка с ключом. Распаковка.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 10А
Класс = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="10-22", name=ФИО, klass=Класс)

## Разбираемся

### Список как инструмент, а не как тема

Списки вы знаете с основной школы. В десятом классе важно другое:
какие операции **дёшевы**, какие **дороги** и что происходит
с памятью, когда список копируют.

### Методы списка

| Метод | Что делает | Возвращает |
|---|---|---|
| `.append(x)` | добавить в конец | `None` |
| `.insert(i, x)` | вставить по индексу | `None` |
| `.extend(другой)` | добавить все элементы другого | `None` |
| `.remove(x)` | удалить **первое** вхождение | `None` |
| `.pop()` | удалить и вернуть последний | элемент |
| `.pop(i)` | удалить и вернуть по индексу | элемент |
| `.index(x)` | индекс первого вхождения | число |
| `.count(x)` | сколько раз встречается | число |
| `.sort()` | отсортировать **на месте** | `None` |
| `.reverse()` | перевернуть на месте | `None` |
| `.copy()` | поверхностная копия | новый список |

Третий столбец важнее первых двух. Методы, меняющие список
на месте, возвращают `None` — поэтому `числа = числа.sort()`
уничтожает данные. Правильно: либо `числа.sort()`, либо
`новый = sorted(числа)`.

### append против extend

```python
a = [1, 2]
a.append([3, 4])    # [1, 2, [3, 4]] — вложенный список
b = [1, 2]
b.extend([3, 4])    # [1, 2, 3, 4]   — элементы по одному
```

### Что дорого, а что дёшево

| Операция | Стоимость |
|---|---|
| взять по индексу, `append`, `pop()` | мгновенно |
| `in`, `.index`, `.count`, `.remove` | перебор всего списка |
| `insert(0, x)`, `pop(0)` | сдвиг всех элементов |
| `sort` | $n \log n$ |

Отсюда практическое правило: **список хорош с конца**. Если нужно
часто добавлять и убирать в начале — это другая структура данных
(о ней в одиннадцатом классе), а проверять принадлежность
лучше множеством.

### Срезы

```python
a[2:5]      # элементы 2, 3, 4
a[:3]       # первые три
a[-2:]      # последние два
a[::2]      # каждый второй
a[::-1]     # перевёрнутая копия
```

Срез всегда создаёт **новый список**. Поэтому `a[::-1]` — это
не то же самое, что `a.reverse()`: первое возвращает копию,
второе переворачивает оригинал и возвращает `None`.

### Копирование и ссылки

```python
a = [1, 2, 3]
b = a           # то же самое имя для того же списка
b.append(4)
print(a)        # [1, 2, 3, 4] — изменился и «первый»

c = a.copy()    # настоящая копия
c.append(5)
print(a)        # без изменений
```

Присваивание списка **не копирует** его. Это та же история, что
с аргументами функций на прошлом уроке: имён два, объект один.

Копию делают тремя равноправными способами: `a.copy()`, `a[:]`,
`list(a)`.

### Генераторы списков

```python
квадраты = [x * x for x in range(1, 6)]
чётные = [x for x in числа if x % 2 == 0]
пары = [(i, j) for i in range(3) for j in range(3)]
```

Читается справа налево: «для каждого x из … если … взять …».
Генератор короче цикла и обычно быстрее, но он хорош только тогда,
когда помещается в одну строку. Трёхэтажный генератор с двумя
условиями — это уже не «короче», а «непонятно».

### Список списков и коварство умножения

```python
таблица = [[0] * 3] * 3      # ловушка!
таблица[0][0] = 1
print(таблица)               # [[1,0,0],[1,0,0],[1,0,0]]
```

Умножение списка **копирует ссылку**, а не содержимое: получилось
три имени одной и той же строки. Правильно — генератором:

```python
таблица = [[0] * 3 for _ in range(3)]
```

Эту ловушку подробно разберём на следующем уроке, но знать
про неё надо уже сейчас.

## Смотрим, как это работает

### Пример 1. Методы, которые возвращают None

In [ ]:
числа = [3, 1, 2]

результат = числа.sort()
print("Вернул sort():", результат)
print("А список стал:", числа)

копия = sorted([3, 1, 2])
print("sorted() вернул:", копия)

### Пример 2. append против extend

In [ ]:
a = [1, 2]
a.append([3, 4])

b = [1, 2]
b.extend([3, 4])

print("append:", a, "— длина", len(a))
print("extend:", b, "— длина", len(b))

### Пример 3. Присваивание не копирует

In [ ]:
исходный = [1, 2, 3]
ссылка = исходный
копия = исходный.copy()

ссылка.append(99)
копия.append(0)

print("Исходный:", исходный)
print("Ссылка:  ", ссылка)
print("Копия:   ", копия)
print("Это один объект:", исходный is ссылка, "| а это разные:", исходный is копия)

### Пример 4. Срезы

In [ ]:
данные = list(range(10))

print("Весь список: ", данные)
print("data[2:5]:   ", данные[2:5])
print("data[:3]:    ", данные[:3])
print("data[-3:]:   ", данные[-3:])
print("data[::2]:   ", данные[::2])
print("data[::-1]:  ", данные[::-1])
print("Оригинал цел:", данные)

### Пример 5. Что дорого

In [ ]:
import time

n = 50_000

начало = time.perf_counter()
в_конец = []
for i in range(n):
    в_конец.append(i)
время_конец = time.perf_counter() - начало

начало = time.perf_counter()
в_начало = []
for i in range(n):
    в_начало.insert(0, i)
время_начало = time.perf_counter() - начало

print(f"append в конец:   {время_конец * 1000:>8.1f} мс")
print(f"insert в начало:  {время_начало * 1000:>8.1f} мс")
print(f"Разница в {время_начало / время_конец:,.0f} раз")

Каждая вставка в начало сдвигает весь список — отсюда и разница.

### Пример 6. Генераторы списков

In [ ]:
числа = list(range(1, 11))

квадраты = [x * x for x in числа]
чётные = [x for x in числа if x % 2 == 0]
подписи = [f"№{x}" for x in числа[:3]]

print("Квадраты:", квадраты)
print("Чётные:  ", чётные)
print("Подписи: ", подписи)

# то же самое обычным циклом
вручную = []
for x in числа:
    if x % 2 == 0:
        вручную.append(x)
print("Совпало:", вручную == чётные)

### Пример 7. Ловушка умножения

In [ ]:
плохо = [[0] * 3] * 3
плохо[0][0] = 1

хорошо = [[0] * 3 for _ in range(3)]
хорошо[0][0] = 1

print("Через умножение:", плохо)
print("Через генератор:", хорошо)

## Пробуем сами

### Задача 1. Длина вложенного списка

Чему равно `len([1, [2, 3]])`?

In [ ]:
#@title 🧩 Задача 1. Длина списка { display-mode: "form" }
#@markdown Впиши число
длина = 0 #@param {type:"integer"}

si.ответ("1", длина, "d4735e3a265e16ee",
         hint="Вложенный список — это один элемент.")

### Задача 2. Метод добавления в конец

Каким методом добавляют **один** элемент в конец списка?

In [ ]:
#@title 🧩 Задача 2. Метод добавления { display-mode: "form" }
#@markdown Впиши название без скобок
метод = "" #@param {type:"string"}

si.ответ("2", метод, "a3021aea224ee968",
         hint="Его сосед extend добавляет сразу много.")

### Задача 3. Отбор генератором

Функция возвращает список элементов, которые больше порога.
Напишите генератором списка в одну строку.

In [ ]:
def больше(числа, порог):
    return ...

In [ ]:
si.check("3", больше, [
    (([1, 5, 9, 3], 4), [5, 9]),
    (([1, 2], 10), []),
    (([7, 7], 6), [7, 7]),
    (([], 0), []),
])

### Задача 4. Копия без исходника

Функция должна вернуть **отсортированную копию** списка,
не меняя исходный.

Осторожно с `.sort()` — он меняет оригинал и возвращает `None`.

In [ ]:
def отсортированная_копия(числа):
    return ...

In [ ]:
si.check("4", отсортированная_копия, [
    ([3, 1, 2], [1, 2, 3]),
    ([5], [5]),
    ([], []),
    ([2, 2, 1], [1, 2, 2]),
])

### Задача 5. Каждый второй с конца

Функция возвращает список, перевёрнутый задом наперёд, в котором
взят каждый второй элемент.

Решается одним срезом.

In [ ]:
def через_один_наоборот(данные):
    return ...

In [ ]:
si.check("5", через_один_наоборот, [
    ([1, 2, 3, 4, 5], [5, 3, 1]),
    ([1, 2], [2]),
    ([], []),
    ([7], [7]),
])

### Задача 6. Таблица нужного размера

Функция возвращает таблицу (список списков) из `строк` строк
и `столбцов` столбцов, заполненную нулями.

Проверка специально меняет один элемент — если попадётесь
на ловушку умножения, это будет видно.

In [ ]:
def пустая_таблица(строк, столбцов):
    return ...

In [ ]:
def меняем_один(строк, столбцов):
    таблица = пустая_таблица(строк, столбцов)
    таблица[0][0] = 1
    return таблица


si.check("6", меняем_один, [
    ((2, 2), [[1, 0], [0, 0]]),
    ((3, 1), [[1], [0], [0]]),
    ((1, 3), [[1, 0, 0]]),
])

## Домашнее задание

### Домашнее задание 1. Замеры операций

Замерьте время трёх операций на списке из 100 000 элементов:
`append` в конец, `insert(0, x)` в начало и проверку `in`
для элемента, которого в списке нет.

Объясните, почему разница именно такая. Покажите замеры учителю.

### Домашнее задание 2. Без повторов, по порядку

Функция возвращает список без повторов, сохраняя порядок первого
появления элементов.

Множество поможет помнить увиденное, но сам ответ — список.

In [ ]:
def без_повторов(данные):
    return ...

In [ ]:
si.check("дз2", без_повторов, [
    ([1, 2, 1, 3, 2], [1, 2, 3]),
    ([5, 5, 5], [5]),
    ([], []),
    ([3, 2, 1], [3, 2, 1]),
])

### Домашнее задание 3. Сколько элементов в срезе

Сколько элементов в срезе `данные[2:5]`, если в списке
десять элементов?

In [ ]:
#@title 🏠 Домашнее 3. Длина среза { display-mode: "form" }
#@markdown Впиши число
в_срезе = 0 #@param {type:"integer"}

si.ответ("дз3", в_срезе, "4e07408562bedb8b",
         hint="Правая граница не входит.")

---

### Любопытно

Почему `insert(0, x)` медленный, а `append` — нет? Потому что
список в памяти — это непрерывный кусок: элементы лежат подряд.
Добавить в конец обычно можно просто заняв следующую ячейку,
а вставка в начало требует сдвинуть всё остальное на шаг вправо.

Зато непрерывность даёт мгновенный доступ по индексу: адрес
элемента вычисляется арифметикой, искать ничего не нужно.
Это классический размен, который встречается в программировании
постоянно: быстро в одном — медленно в другом, и выбирать
приходится под задачу.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 21](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-21.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 23 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-23.ipynb)